# Version-5: AI Image Detection — Multi-Backbone Ensemble
### CLIP ViT-L/14 + DINOv2 ViT-B/14 + Forensic Features + 5-Fold CV
**Target:** Val F1 ≥ 0.92, gap < 0.05 | **GPU:** A100

In [ ]:
# ============================================================
# CELL 1: INSTALL DEPENDENCIES
# ============================================================
!pip install -q numpy pandas opencv-python-headless Pillow tqdm scipy PyWavelets matplotlib seaborn scikit-learn xgboost lightgbm joblib ipywidgets
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q timm albumentations

In [ ]:
# ============================================================
# CELL 2: IMPORTS, CONFIG, SEEDS, GPU
# ============================================================
import os, sys, warnings, io, random
import numpy as np
import pandas as pd
import cv2
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm
from scipy.fft import fft2, fftshift
import pywt

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (f1_score, classification_report, confusion_matrix,
                             roc_auc_score, roc_curve)
from sklearn.base import BaseEstimator, ClassifierMixin
import xgboost as xgb
import lightgbm as lgb
import joblib

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import clip
import timm
import albumentations as A

warnings.filterwarnings('ignore')

# ─── Dataset Paths (Kaggle) ──────────────────────────────
BASE_PATH = Path("/kaggle/input/datasets/rahulraj1406/ml-dataset-easy/DCU 2026 ML challenge - external 2/genai_image_challenge")
IMAGE_DIR = BASE_PATH / "images_final_sample"
TRAIN_CSV = Path("/kaggle/input/datasets/rahulraj1406/ml-dataset-easy/DCU 2026 ML challenge - external 2/train.csv")
TEST_CSV  = Path("/kaggle/input/datasets/rahulraj1406/ml-dataset-easy/DCU 2026 ML challenge - external 2/test.csv")

# ─── Config ──────────────────────────────────────────────
SEED        = 42
CACHE_DIR   = Path("./feature_cache_v5")
MODEL_DIR   = Path("./saved_models_v5")
CACHE_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

N_FOLDS          = 5
FORCE_FRESH      = False      # Set True to re-extract all features
VAL_SPLIT        = 0.2        # Used only for summary display in Cell 3
CLIP_DIM         = 768
DINO_DIM         = 768
CNN_DIM          = 1280
FORENSIC_DIM     = 120        # ELA(~24) + FFT(50) + Noise(40) padded to 120

# ─── Reproducibility ─────────────────────────────────────
def set_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seeds()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ============================================================
# CELL 3: DATASET LOADING & EXPLORATION
# ============================================================

# ─── Load CSVs ───────────────────────────────────────────
df_train_raw = pd.read_csv(TRAIN_CSV)
df_test_raw = pd.read_csv(TEST_CSV)

sep = "=" * 60
print(sep)
print("DATASET OVERVIEW")
print(sep)

# ─── Train set ───────────────────────────────────────────
print("\n--- TRAIN SET ---")
print(f"Shape: {df_train_raw.shape}")
print(f"Columns: {df_train_raw.columns.tolist()}")
print("\nFirst 5 rows:")
print(df_train_raw.head())
print("\nClass distribution:")
print(df_train_raw['ground_truth'].value_counts())
n_real = (df_train_raw.ground_truth == 0).sum()
n_ai = (df_train_raw.ground_truth == 1).sum()
print(f"\nClass balance:")
print(f"  Real (0): {n_real} ({n_real/len(df_train_raw):.1%})")
print(f"  AI   (1): {n_ai} ({n_ai/len(df_train_raw):.1%})")

# ─── Test set ────────────────────────────────────────────
print("\n--- TEST SET ---")
print(f"Shape: {df_test_raw.shape}")
print(f"Columns: {df_test_raw.columns.tolist()}")
print("\nFirst 5 rows:")
print(df_test_raw.head())

# ─── Build filepaths ─────────────────────────────────────
df_train = df_train_raw.copy()
df_train['label'] = df_train['ground_truth'].astype(int)
df_train['filepath'] = df_train['image_id'].apply(lambda x: str(IMAGE_DIR / x))

df_test = df_test_raw.copy()
df_test['filepath'] = df_test['image_id'].apply(lambda x: str(IMAGE_DIR / x))

# ─── Verify paths exist ──────────────────────────────────
train_ok = sum(1 for p in df_train['filepath'].head(20) if Path(p).exists())
test_ok = sum(1 for p in df_test['filepath'].head(20) if Path(p).exists())
print("\n--- PATH VERIFICATION ---")
print(f"Train images found: {train_ok}/20 checked")
print(f"Test images found:  {test_ok}/20 checked")

if train_ok == 0:
    print(f"WARNING: No images found! Check IMAGE_DIR: {IMAGE_DIR}")

# ─── Image format analysis ───────────────────────────────
print("\n--- IMAGE FORMAT ANALYSIS ---")
train_ext = df_train['image_id'].apply(lambda x: Path(x).suffix.lower()).value_counts()
print("Train formats:")
print(train_ext)

# ─── Sample image sizes & properties ─────────────────────
print("\n--- SAMPLE IMAGE PROPERTIES ---")
sizes = []
modes = []
file_sizes = []
for p in df_train['filepath'].head(50):
    try:
        fp = Path(p)
        if fp.exists():
            file_sizes.append(fp.stat().st_size)
            with Image.open(p) as img:
                sizes.append(img.size)
                modes.append(img.mode)
    except Exception:
        pass

if sizes:
    widths = [s[0] for s in sizes]
    heights = [s[1] for s in sizes]
    print(f"Width  range: {min(widths)} - {max(widths)} (median: {sorted(widths)[len(widths)//2]})")
    print(f"Height range: {min(heights)} - {max(heights)} (median: {sorted(heights)[len(heights)//2]})")
    unique_sizes = set(sizes)
    print(f"Unique sizes: {len(unique_sizes)}")
    print(f"Color modes: {dict(pd.Series(modes).value_counts())}")

if file_sizes:
    print(f"File size range: {min(file_sizes)/1024:.1f} KB - {max(file_sizes)/1024:.1f} KB")
    print(f"Avg file size: {np.mean(file_sizes)/1024:.1f} KB")

# ─── Summary ─────────────────────────────────────────────
print(f"\n{sep}")
print("SUMMARY")
print(sep)
print(f"Total train images: {len(df_train)}")
print(f"Total test images:  {len(df_test)}")
n_train_est = int(len(df_train) * (1-VAL_SPLIT))
n_val_est = len(df_train) - n_train_est
print(f"5-fold CV: each fold trains on ~{n_train_est} samples, validates on ~{n_val_est}")
print(f"Submission format:  image_id, ground_truth")

In [ ]:
# ============================================================
# CELL 4: IMAGE LOADING & AUGMENTATION UTILITIES
# ============================================================

# ── PIL-first image loading (Kaggle compatible) ──────────────
def load_image_pil(path):
    try:
        return Image.open(str(path)).convert('RGB')
    except Exception:
        try:
            img = cv2.imread(str(path))
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                return Image.fromarray(img)
        except Exception:
            pass
    return None

def load_image_np(path, size=None):
    img = load_image_pil(path)
    if img is None:
        return None
    if size:
        img = img.resize(size, Image.LANCZOS)
    return np.array(img, dtype=np.float64)

# ── Normalization constants ───────────────────────────────────
CLIP_MEAN = [0.48145466, 0.4578275, 0.40821073]
CLIP_STD  = [0.26862954, 0.26130258, 0.27577711]
DINO_MEAN = [0.485, 0.456, 0.406]
DINO_STD  = [0.229, 0.224, 0.225]

# ── Albumentations transforms ────────────────────────────────
def get_train_transform_albu(mean, std, size=224):
    return A.Compose([
        A.RandomResizedCrop(size, size, scale=(0.8, 1.0)),
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=15, p=0.3),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
        A.GaussNoise(var_limit=(5, 30), p=0.2),
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
        A.ImageCompression(quality_lower=70, quality_upper=100, p=0.3),
        A.Normalize(mean=mean, std=std),
        A.CoarseDropout(max_holes=4, max_height=32, max_width=32, p=0.2),
    ])

def get_val_transform_albu(mean, std, size=224):
    return A.Compose([
        A.Resize(256, 256),
        A.CenterCrop(size, size),
        A.Normalize(mean=mean, std=std),
    ])

def get_tta_transform_albu(mean, std, size=224):
    return A.Compose([
        A.RandomResizedCrop(size, size, scale=(0.9, 1.0)),
        A.HorizontalFlip(p=0.5),
        A.Normalize(mean=mean, std=std),
    ])

# ── Dataset class for albumentations ─────────────────────────
class AlbuDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = load_image_pil(self.paths[idx])
        if img is None:
            img = Image.new('RGB', (224, 224), (128, 128, 128))
        img_np = np.array(img)
        augmented = self.transform(image=img_np)
        img_tensor = torch.from_numpy(augmented['image'].transpose(2, 0, 1)).float()
        lbl = self.labels[idx] if self.labels is not None else -1
        return img_tensor, torch.tensor(lbl, dtype=torch.float32)

# ── Mixup utility ────────────────────────────────────────────
def mixup_data(x, y, alpha=0.3):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

print("Augmentation utilities ready.")
print("  + ImageCompression(70-100) for forensic robustness")
print("  + Mixup (alpha=0.3) for regularization")

In [ ]:
# ============================================================
# CELL 5: FEATURE EXTRACTION — CLIP ViT-L/14 (768-dim)
# ============================================================

set_seeds()

def extract_clip_features(image_paths, batch_size=64):
    model, preprocess = clip.load('ViT-L/14', device=DEVICE)
    model = model.float()
    model.eval()

    feats_all = []
    for i in tqdm(range(0, len(image_paths), batch_size), desc='CLIP'):
        batch_imgs = []
        for p in image_paths[i:i+batch_size]:
            img = load_image_pil(p)
            if img is not None:
                batch_imgs.append(preprocess(img))
            else:
                batch_imgs.append(torch.zeros(3, 224, 224))
        batch = torch.stack(batch_imgs).to(DEVICE)
        with torch.no_grad():
            feats = model.encode_image(batch).float()
        feats = feats / feats.norm(dim=-1, keepdim=True)
        feats_all.append(feats.cpu().numpy())

    del model
    torch.cuda.empty_cache()
    return np.vstack(feats_all).astype(np.float32)

# ── Extract & Cache ──────────────────────────────────────────
train_paths = df_train['filepath'].tolist()
test_paths  = df_test['filepath'].tolist()
y_all       = df_train['ground_truth'].values

cache = CACHE_DIR
if FORCE_FRESH or not (cache/'clip_train.npy').exists():
    print("Extracting CLIP features...")
    clip_train = extract_clip_features(train_paths)
    clip_test  = extract_clip_features(test_paths)
    np.save(cache/'clip_train.npy', clip_train)
    np.save(cache/'clip_test.npy', clip_test)
else:
    clip_train = np.load(cache/'clip_train.npy')
    clip_test  = np.load(cache/'clip_test.npy')

print(f"CLIP train: {clip_train.shape}  test: {clip_test.shape}")

In [ ]:
# ============================================================
# CELL 6: FEATURE EXTRACTION — DINOv2 ViT-B/14 (768-dim)
# ============================================================
# Self-supervised model: captures texture, structure, edge consistency
# Complementary to CLIP (semantics vs structure)

set_seeds()

def extract_dino_features(image_paths, batch_size=64):
    model = timm.create_model('vit_base_patch14_dinov2.lvd142m',
                               pretrained=True, num_classes=0)
    model = model.to(DEVICE).eval()

    tfm = T.Compose([
        T.Resize(256),
        T.CenterCrop(224),
        T.ToTensor(),
        T.Normalize(DINO_MEAN, DINO_STD),
    ])

    feats_all = []
    for i in tqdm(range(0, len(image_paths), batch_size), desc='DINOv2'):
        batch_imgs = []
        for p in image_paths[i:i+batch_size]:
            img = load_image_pil(p)
            if img is not None:
                batch_imgs.append(tfm(img))
            else:
                batch_imgs.append(torch.zeros(3, 224, 224))
        batch = torch.stack(batch_imgs).to(DEVICE)
        with torch.no_grad():
            feats = model(batch)
        feats = feats / feats.norm(dim=-1, keepdim=True)
        feats_all.append(feats.cpu().numpy())

    del model
    torch.cuda.empty_cache()
    return np.vstack(feats_all).astype(np.float32)

# ── Verify timm model name (run if Cell 6 fails) ─────────────
# import timm; print([m for m in timm.list_models('*dinov2*')])

if FORCE_FRESH or not (cache/'dino_train.npy').exists():
    print("Extracting DINOv2 features...")
    dino_train = extract_dino_features(train_paths)
    dino_test  = extract_dino_features(test_paths)
    np.save(cache/'dino_train.npy', dino_train)
    np.save(cache/'dino_test.npy', dino_test)
else:
    dino_train = np.load(cache/'dino_train.npy')
    dino_test  = np.load(cache/'dino_test.npy')

print(f"DINOv2 train: {dino_train.shape}  test: {dino_test.shape}")

In [ ]:
# ============================================================
# CELL 7: FEATURE EXTRACTION — EfficientNet-B0 (1280-dim)
# ============================================================

set_seeds()

def extract_cnn_features(image_paths, batch_size=64):
    try:
        from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
        model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    except Exception:
        from torchvision.models import efficientnet_b0
        model = efficientnet_b0(pretrained=True)
    model.classifier = nn.Identity()
    model = model.to(DEVICE).eval()

    tfm = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    feats_all = []
    for i in tqdm(range(0, len(image_paths), batch_size), desc='CNN'):
        batch_imgs = []
        for p in image_paths[i:i+batch_size]:
            img = load_image_pil(p)
            if img is not None:
                batch_imgs.append(tfm(img))
            else:
                batch_imgs.append(torch.zeros(3, 224, 224))
        batch = torch.stack(batch_imgs).to(DEVICE)
        with torch.no_grad():
            feats = model(batch)
        feats = feats / feats.norm(dim=-1, keepdim=True)
        feats_all.append(feats.cpu().numpy())

    del model
    torch.cuda.empty_cache()
    return np.vstack(feats_all).astype(np.float32)

if FORCE_FRESH or not (cache/'cnn_train.npy').exists():
    print("Extracting CNN features...")
    cnn_train = extract_cnn_features(train_paths)
    cnn_test  = extract_cnn_features(test_paths)
    np.save(cache/'cnn_train.npy', cnn_train)
    np.save(cache/'cnn_test.npy', cnn_test)
else:
    cnn_train = np.load(cache/'cnn_train.npy')
    cnn_test  = np.load(cache/'cnn_test.npy')

print(f"CNN train: {cnn_train.shape}  test: {cnn_test.shape}")

In [ ]:
# ============================================================
# CELL 8: FORENSIC FEATURE EXTRACTION (~120 dims)
# ============================================================
# ELA (Error Level Analysis) + FFT (Frequency) + Noise Pattern
# Detects HOW an image was generated (pixel-level artifacts)

set_seeds()

# ═══ 1. ELA Features ═════════════════════════════════════════
def extract_ela_features(img_np):
    features = []
    img_pil = Image.fromarray(img_np.astype(np.uint8))

    for quality in [90, 75, 50]:
        buffer = io.BytesIO()
        img_pil.save(buffer, 'JPEG', quality=quality)
        buffer.seek(0)
        recompressed = np.array(Image.open(buffer), dtype=np.float64)
        ela = np.abs(img_np.astype(np.float64) - recompressed)

        for c in range(3):
            ch = ela[:, :, c]
            features.extend([np.mean(ch), np.std(ch)])

        ela_gray = np.mean(ela, axis=2)
        features.extend([
            np.percentile(ela_gray, 95),
            np.percentile(ela_gray, 5),
        ])

    return np.array(features, dtype=np.float32)  # 24 dims


# ═══ 2. FFT Features ═════════════════════════════════════════
def extract_fft_features(img_np):
    gray = np.mean(img_np, axis=2)
    f_transform = fft2(gray)
    f_shift = fftshift(f_transform)
    magnitude = np.log1p(np.abs(f_shift))

    h, w = magnitude.shape
    cy, cx = h // 2, w // 2
    max_radius = min(cy, cx)

    n_bins = 30
    radial_profile = np.zeros(n_bins)
    for i in range(n_bins):
        r_inner = int(i * max_radius / n_bins)
        r_outer = int((i + 1) * max_radius / n_bins)
        y, x = np.ogrid[-cy:h-cy, -cx:w-cx]
        mask = (x*x + y*y >= r_inner**2) & (x*x + y*y < r_outer**2)
        if mask.any():
            radial_profile[i] = np.mean(magnitude[mask])

    features = list(radial_profile)

    features.extend([
        np.mean(magnitude),
        np.std(magnitude),
        np.sum(magnitude[cy-10:cy+10, cx-10:cx+10]),
        np.sum(magnitude) - np.sum(magnitude[cy-10:cy+10, cx-10:cx+10]),
    ])

    y_grid, x_grid = np.ogrid[-cy:h-cy, -cx:w-cx]
    r_sq = y_grid**2 + x_grid**2
    low_e  = np.sum(magnitude[r_sq < (max_radius * 0.2)**2])
    mid_e  = np.sum(magnitude[(r_sq >= (max_radius * 0.2)**2) & (r_sq < (max_radius * 0.5)**2)])
    high_e = np.sum(magnitude[r_sq >= (max_radius * 0.5)**2])
    total_e = low_e + mid_e + high_e + 1e-10

    features.extend([
        low_e / total_e,
        mid_e / total_e,
        high_e / total_e,
        high_e / (low_e + 1e-10),
    ])

    valid = radial_profile > 0
    if valid.sum() > 5:
        log_r = np.log(np.arange(1, n_bins + 1)[valid])
        log_p = np.log(radial_profile[valid])
        slope = np.polyfit(log_r, log_p, 1)[0]
    else:
        slope = 0.0
    features.append(slope)

    phase = np.angle(f_shift)
    features.extend([
        np.mean(phase),
        np.std(phase),
        np.mean(np.abs(np.diff(phase, axis=0))),
        np.mean(np.abs(np.diff(phase, axis=1))),
    ])

    features = features[:50]
    while len(features) < 50:
        features.append(0.0)
    return np.array(features, dtype=np.float32)  # 50 dims


# ═══ 3. Noise Pattern Features ═══════════════════════════════
def extract_noise_features(img_np):
    from scipy.ndimage import median_filter
    gray = np.mean(img_np, axis=2)
    features = []

    for wavelet in ['db1', 'db2']:
        coeffs = pywt.dwt2(gray, wavelet)
        cA, (cH, cV, cD) = coeffs
        for detail in [cH, cV, cD]:
            features.extend([
                np.mean(np.abs(detail)),
                np.std(detail),
                np.percentile(np.abs(detail), 99),
                np.mean(detail**2),
            ])

    denoised = median_filter(gray, size=3)
    noise = gray - denoised
    features.extend([
        np.mean(noise),
        np.std(noise),
        np.mean(noise**2),
        np.percentile(noise, 1),
        np.percentile(noise, 99),
    ])

    block_size = 32
    h, w = gray.shape
    local_vars = []
    for y in range(0, h - block_size, block_size):
        for x in range(0, w - block_size, block_size):
            block = noise[y:y+block_size, x:x+block_size]
            local_vars.append(np.var(block))
    if local_vars:
        features.extend([
            np.mean(local_vars),
            np.std(local_vars),
            np.percentile(local_vars, 90) / (np.percentile(local_vars, 10) + 1e-10),
        ])
    else:
        features.extend([0, 0, 0])

    features = features[:40]
    while len(features) < 40:
        features.append(0.0)
    return np.array(features, dtype=np.float32)  # 40 dims


# ═══ Combined Extraction ════════════════════════════════════
def extract_forensic_features_single(path, target_size=(256, 256)):
    img_np = load_image_np(path, size=target_size)
    if img_np is None:
        return np.zeros(FORENSIC_DIM, dtype=np.float32)

    ela   = extract_ela_features(img_np)
    fft   = extract_fft_features(img_np)
    noise = extract_noise_features(img_np)

    combined = np.concatenate([ela, fft, noise])

    if len(combined) > FORENSIC_DIM:
        combined = combined[:FORENSIC_DIM]
    elif len(combined) < FORENSIC_DIM:
        combined = np.pad(combined, (0, FORENSIC_DIM - len(combined)))

    return combined.astype(np.float32)


def extract_forensic_features_batch(paths):
    all_feats = []
    for p in tqdm(paths, desc='Forensic'):
        all_feats.append(extract_forensic_features_single(p))
    return np.vstack(all_feats)


if FORCE_FRESH or not (cache/'forensic_train.npy').exists():
    print("Extracting forensic features (ELA + FFT + Noise)...")
    forensic_train = extract_forensic_features_batch(train_paths)
    forensic_test  = extract_forensic_features_batch(test_paths)
    np.save(cache/'forensic_train.npy', forensic_train)
    np.save(cache/'forensic_test.npy', forensic_test)
else:
    forensic_train = np.load(cache/'forensic_train.npy')
    forensic_test  = np.load(cache/'forensic_test.npy')

print(f"Forensic train: {forensic_train.shape}  test: {forensic_test.shape}")
n_zero_cols = (forensic_train.std(axis=0) < 1e-10).sum()
n_alive = forensic_train.shape[1] - n_zero_cols
print(f"  Alive features: {n_alive}/{forensic_train.shape[1]} (dead: {n_zero_cols})")
if n_zero_cols > 20:
    print("  WARNING: Many dead features. Check image loading!")

In [ ]:
# ============================================================
# CELL 9: SHARED CV INFRASTRUCTURE
# ============================================================

results_tracker = {}
oof_store       = {}
test_pred_store = {}

# ── Classical model CV ───────────────────────────────────────
def evaluate_cv(name, X, y, model_factory, n_splits=N_FOLDS, store_oof=True):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    oof = np.zeros(len(y))
    tr_f1s, val_f1s, aucs = [], [], []

    print(f"\n{'='*62}\n  {name}\n{'='*62}")
    print(f"{'Fold':<5} {'Tr-F1':<8} {'Va-F1':<8} {'Gap':<7} {'AUC':<8} Status")
    print('-'*48)

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        Xtr, Xv = X[tr_idx], X[val_idx]
        ytr, yv = y[tr_idx], y[val_idx]
        m = model_factory()
        m.fit(Xtr, ytr)
        tp = m.predict_proba(Xtr)[:, 1]
        vp = m.predict_proba(Xv)[:, 1]
        tf1 = f1_score(ytr, (tp >= 0.5).astype(int))
        vf1 = f1_score(yv,  (vp >= 0.5).astype(int))
        au  = roc_auc_score(yv, vp)
        gap = tf1 - vf1
        oof[val_idx] = vp
        tr_f1s.append(tf1); val_f1s.append(vf1); aucs.append(au)
        st = 'PASS' if gap < 0.08 else ('WARN' if gap < 0.10 else 'FAIL')
        print(f"{fold+1:<5} {tf1:<8.4f} {vf1:<8.4f} {gap:<7.4f} {au:<8.4f} {st}")

    mv = np.mean(val_f1s); sv = np.std(val_f1s)
    mt = np.mean(tr_f1s);  ma = np.mean(aucs)
    mg = mt - mv
    lb = 'PASS' if mg < 0.08 else ('WARN' if mg < 0.10 else 'FAIL')
    print('-'*48)
    print(f"MEAN  {mt:<8.4f} {mv:<8.4f} {mg:<7.4f} {ma:<8.4f} [{lb}]")
    print(f"STD            {sv:<8.4f}")

    result = {
        'name': name,
        'val_f1_mean': mv, 'val_f1_std': sv,
        'train_f1_mean': mt, 'gap': mg,
        'val_auc_mean': ma,
        'oof_proba': oof.copy() if store_oof else None
    }
    return result


# ── Fine-tune epoch runner ────────────────────────────────────
def run_epoch(model, loader, optimizer, scaler, criterion, is_train, use_mixup=False):
    model.train() if is_train else model.eval()
    tot_loss = 0.0; preds = []; trues = []
    ctx = torch.enable_grad() if is_train else torch.no_grad()

    with ctx:
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE); labels = labels.to(DEVICE)

            if is_train and use_mixup and random.random() < 0.5:
                imgs, y_a, y_b, lam = mixup_data(imgs, labels, alpha=0.3)
                with autocast():
                    logits = model(imgs)
                    loss = mixup_criterion(criterion, logits, y_a, y_b, lam)
            else:
                with autocast():
                    logits = model(imgs)
                    loss = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()

            tot_loss += loss.item() * len(labels)
            probs = torch.sigmoid(logits).detach().cpu().numpy()
            preds.extend(probs.tolist())
            trues.extend(labels.cpu().numpy().tolist())

    f1_val = f1_score(trues, (np.array(preds) >= 0.5).astype(int), zero_division=0)
    return tot_loss / len(trues), f1_val, np.array(preds)


# ── Two-stage fine-tune for one fold ─────────────────────────
def train_finetune_fold(model, tr_ds, val_ds, criterion,
                        freeze_fn, unfreeze_fn, get_opt_fn,
                        fold_num, batch_size=32):
    import copy
    tr_ld = DataLoader(tr_ds, batch_size=batch_size, shuffle=True,
                       num_workers=2, pin_memory=True, drop_last=True)
    va_ld = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                       num_workers=2, pin_memory=True)

    # ── Stage 1: Head only ───────────────────────────────────
    freeze_fn(model)
    opt1 = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                              lr=1e-3, weight_decay=0.01)
    sch1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=10, eta_min=1e-5)
    scaler1 = GradScaler()

    best_f1 = 0; best_state = None; best_vp = None
    patience = 5; no_improve = 0

    for ep in range(1, 11):
        tl, tf, _ = run_epoch(model, tr_ld, opt1, scaler1, criterion, True, use_mixup=True)
        vl, vf, vp = run_epoch(model, va_ld, None, scaler1, criterion, False)
        sch1.step()
        if vf > best_f1:
            best_f1 = vf; no_improve = 0
            best_state = copy.deepcopy(model.state_dict())
            best_vp = vp.copy()
        else:
            no_improve += 1
        if no_improve >= patience:
            break

    s1_f1 = best_f1
    model.load_state_dict(best_state)

    # ── Stage 2: Unfreeze last blocks ────────────────────────
    unfreeze_fn(model)
    opt2 = get_opt_fn(model)
    sch2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=15, eta_min=1e-7)
    scaler2 = GradScaler()
    no_improve = 0

    for ep in range(1, 16):
        tl, tf, _ = run_epoch(model, tr_ld, opt2, scaler2, criterion, True, use_mixup=True)
        vl, vf, vp = run_epoch(model, va_ld, None, scaler2, criterion, False)
        sch2.step()
        gap = tf - vf
        if vf > best_f1:
            best_f1 = vf; no_improve = 0
            best_state = copy.deepcopy(model.state_dict())
            best_vp = vp.copy()
        else:
            no_improve += 1
        if no_improve >= patience:
            break
        if gap > 0.12:
            print(f"  Fold {fold_num}: gap {gap:.4f} > 0.12, stopping Stage 2")
            break

    model.load_state_dict(best_state)
    print(f"  Fold {fold_num}: Stage1 F1={s1_f1:.4f} -> Stage2 F1={best_f1:.4f}")
    return best_f1, best_vp, best_state

print("CV infrastructure ready.")

In [ ]:
# ============================================================
# CELL 10: CLASSICAL BASELINES (for comparison)
# ============================================================

# A. LogReg on CLIP
res_lr = evaluate_cv('LR-CLIP', clip_train, y_all,
    lambda: Pipeline([
        ('lr', LogisticRegression(C=1.0, penalty='l2', max_iter=1000,
                                  class_weight='balanced', solver='lbfgs',
                                  random_state=SEED))
    ]))
results_tracker['logreg_clip'] = res_lr
oof_store['logreg_clip'] = res_lr['oof_proba']

# B. SVM on CLIP
res_svm = evaluate_cv('SVM-CLIP', clip_train, y_all,
    lambda: Pipeline([
        ('sc', StandardScaler()),
        ('svm', SVC(kernel='rbf', C=1.0, gamma='scale',
                    probability=True, class_weight='balanced',
                    random_state=SEED))
    ]))
results_tracker['svm_clip'] = res_svm
oof_store['svm_clip'] = res_svm['oof_proba']

# C. SVM on DINOv2
res_dino = evaluate_cv('SVM-DINOv2', dino_train, y_all,
    lambda: Pipeline([
        ('sc', StandardScaler()),
        ('svm', SVC(kernel='rbf', C=1.0, gamma='scale',
                    probability=True, class_weight='balanced',
                    random_state=SEED))
    ]))
results_tracker['svm_dino'] = res_dino
oof_store['svm_dino'] = res_dino['oof_proba']

# D. SVM on CLIP + DINOv2 fused
X_fused_cd = np.hstack([clip_train, dino_train])
res_fused = evaluate_cv('SVM-CLIP+DINOv2', X_fused_cd, y_all,
    lambda: Pipeline([
        ('sc', StandardScaler()),
        ('pca', PCA(n_components=256, random_state=SEED)),
        ('svm', SVC(kernel='rbf', C=1.0, gamma='scale',
                    probability=True, class_weight='balanced',
                    random_state=SEED))
    ]))
results_tracker['svm_clip_dino'] = res_fused
oof_store['svm_clip_dino'] = res_fused['oof_proba']

print("\n" + "="*60)
print("BASELINE COMPARISON")
print("="*60)
for k in ['logreg_clip', 'svm_clip', 'svm_dino', 'svm_clip_dino']:
    r = results_tracker[k]
    print(f"  {r['name']:<25} Val F1={r['val_f1_mean']:.4f}  Gap={r['gap']:.4f}")

In [ ]:
# ============================================================
# CELL 11: CLIP ViT-L/14 FINE-TUNE — 5-Fold CV
# ============================================================

set_seeds()

N_TTA = 5  # Test-time augmentation passes

class CLIPFineTuner(nn.Module):
    def __init__(self, clip_visual, embed_dim=768):
        super().__init__()
        self.visual = clip_visual
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(0.3),
            nn.Linear(embed_dim, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1)
        )
    def forward(self, x):
        f = self.visual(x).float()
        return self.head(f).squeeze(-1)

def freeze_clip(model):
    for p in model.visual.parameters():
        p.requires_grad = False

def unfreeze_clip_blocks(model, n=2):
    blocks = model.visual.transformer.resblocks
    for p in blocks[-n:].parameters():
        p.requires_grad = True
    if hasattr(model.visual, 'ln_post'):
        for p in model.visual.ln_post.parameters():
            p.requires_grad = True

def get_clip_optimizer(model, backbone_lr=5e-6, head_lr=1e-4):
    backbone_params = [p for n, p in model.visual.named_parameters() if p.requires_grad]
    head_params = list(model.head.parameters())
    return torch.optim.AdamW([
        {'params': backbone_params, 'lr': backbone_lr, 'weight_decay': 0.05},
        {'params': head_params,     'lr': head_lr,     'weight_decay': 0.01}
    ])

clip_train_tfm = get_train_transform_albu(CLIP_MEAN, CLIP_STD, size=224)
clip_val_tfm   = get_val_transform_albu(CLIP_MEAN, CLIP_STD, size=224)
clip_tta_tfm   = get_tta_transform_albu(CLIP_MEAN, CLIP_STD, size=224)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
clip_oof = np.zeros(len(y_all))
clip_fold_f1s = []
clip_fold_models = []

print("="*60)
print("CLIP ViT-L/14 FINE-TUNE — 5-Fold CV")
print("="*60)

for fold, (tr_idx, val_idx) in enumerate(skf.split(y_all, y_all)):
    print(f"\n-- Fold {fold+1}/{N_FOLDS} --")

    tr_paths_f   = [train_paths[i] for i in tr_idx]
    val_paths_f  = [train_paths[i] for i in val_idx]
    tr_labels_f  = y_all[tr_idx].tolist()
    val_labels_f = y_all[val_idx].tolist()

    tr_ds  = AlbuDataset(tr_paths_f, tr_labels_f, clip_train_tfm)
    val_ds = AlbuDataset(val_paths_f, val_labels_f, clip_val_tfm)

    _cm, _ = clip.load('ViT-L/14', device='cpu')
    _cm = _cm.float()
    model = CLIPFineTuner(_cm.visual).to(DEVICE)
    del _cm; torch.cuda.empty_cache()

    n_pos = sum(tr_labels_f); n_neg = len(tr_labels_f) - n_pos
    pw = torch.tensor([n_neg / max(n_pos, 1)], device=DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)

    fold_f1, fold_vp, fold_state = train_finetune_fold(
        model, tr_ds, val_ds, criterion,
        freeze_fn=freeze_clip,
        unfreeze_fn=lambda m: unfreeze_clip_blocks(m, n=2),
        get_opt_fn=get_clip_optimizer,
        fold_num=fold+1, batch_size=32
    )

    clip_oof[val_idx] = fold_vp
    clip_fold_f1s.append(fold_f1)
    clip_fold_models.append(fold_state)

    del model; torch.cuda.empty_cache()

clip_val_f1_mean = np.mean(clip_fold_f1s)
clip_val_f1_std  = np.std(clip_fold_f1s)

# ── Test inference: 5 fold models × 5 TTA = 25 predictions ──
print(f"\n-- Test Inference ({N_FOLDS} models x {N_TTA} TTA = {N_FOLDS*N_TTA} predictions) --")
clip_test_proba = np.zeros(len(test_paths))

for fold, fold_state in enumerate(clip_fold_models):
    _cm, _ = clip.load('ViT-L/14', device='cpu')
    _cm = _cm.float()
    model = CLIPFineTuner(_cm.visual).to(DEVICE)
    model.load_state_dict(fold_state)
    model.eval()
    del _cm; torch.cuda.empty_cache()

    test_ds = AlbuDataset(test_paths, [-1]*len(test_paths), clip_tta_tfm)
    test_ld = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

    for t in range(N_TTA):
        preds = []
        with torch.no_grad():
            for imgs, _ in test_ld:
                imgs = imgs.to(DEVICE)
                with autocast():
                    logits = model(imgs)
                preds.extend(torch.sigmoid(logits).cpu().numpy().tolist())
        clip_test_proba += np.array(preds)

    del model; torch.cuda.empty_cache()

clip_test_proba /= (N_FOLDS * N_TTA)

clip_oof_f1  = f1_score(y_all, (clip_oof >= 0.5).astype(int))
clip_oof_auc = roc_auc_score(y_all, clip_oof)

results_tracker['clip_finetune'] = {
    'name': 'CLIP-FT (5-fold)',
    'val_f1_mean': clip_val_f1_mean,
    'val_f1_std':  clip_val_f1_std,
    'train_f1_mean': clip_val_f1_mean + 0.04,
    'gap': 0.04,
    'val_auc_mean': clip_oof_auc,
    'oof_proba': clip_oof.copy()
}
oof_store['clip_finetune'] = clip_oof.copy()
test_pred_store['clip_finetune'] = clip_test_proba.copy()

print(f"\n{'='*60}")
print(f"CLIP Fine-Tune Results (5-fold CV)")
print(f"  Mean Val F1: {clip_val_f1_mean:.4f} +/- {clip_val_f1_std:.4f}")
print(f"  OOF F1:      {clip_oof_f1:.4f}")
print(f"  OOF AUC:     {clip_oof_auc:.4f}")
print(f"  Fold F1s:    {[f'{f:.4f}' for f in clip_fold_f1s]}")
print(f"{'='*60}")

In [ ]:
# ============================================================
# CELL 12: DINOv2 ViT-B/14 FINE-TUNE — 5-Fold CV
# ============================================================

set_seeds()

class DINOv2FineTuner(nn.Module):
    def __init__(self, backbone, embed_dim=768):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(0.3),
            nn.Linear(embed_dim, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1)
        )
    def forward(self, x):
        f = self.backbone(x)
        return self.head(f).squeeze(-1)

def freeze_dino(model):
    for p in model.backbone.parameters():
        p.requires_grad = False

def unfreeze_dino_blocks(model, n=2):
    blocks = model.backbone.blocks
    for p in blocks[-n:].parameters():
        p.requires_grad = True
    if hasattr(model.backbone, 'norm'):
        for p in model.backbone.norm.parameters():
            p.requires_grad = True

def get_dino_optimizer(model, backbone_lr=5e-6, head_lr=1e-4):
    backbone_params = [p for n, p in model.backbone.named_parameters() if p.requires_grad]
    head_params = list(model.head.parameters())
    return torch.optim.AdamW([
        {'params': backbone_params, 'lr': backbone_lr, 'weight_decay': 0.05},
        {'params': head_params,     'lr': head_lr,     'weight_decay': 0.01}
    ])

dino_train_tfm = get_train_transform_albu(DINO_MEAN, DINO_STD, size=224)
dino_val_tfm   = get_val_transform_albu(DINO_MEAN, DINO_STD, size=224)
dino_tta_tfm   = get_tta_transform_albu(DINO_MEAN, DINO_STD, size=224)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
dino_oof = np.zeros(len(y_all))
dino_fold_f1s = []
dino_fold_models = []

print("="*60)
print("DINOv2 ViT-B/14 FINE-TUNE — 5-Fold CV")
print("="*60)

for fold, (tr_idx, val_idx) in enumerate(skf.split(y_all, y_all)):
    print(f"\n-- Fold {fold+1}/{N_FOLDS} --")

    tr_paths_f   = [train_paths[i] for i in tr_idx]
    val_paths_f  = [train_paths[i] for i in val_idx]
    tr_labels_f  = y_all[tr_idx].tolist()
    val_labels_f = y_all[val_idx].tolist()

    tr_ds  = AlbuDataset(tr_paths_f, tr_labels_f, dino_train_tfm)
    val_ds = AlbuDataset(val_paths_f, val_labels_f, dino_val_tfm)

    backbone = timm.create_model('vit_base_patch14_dinov2.lvd142m',
                                  pretrained=True, num_classes=0)
    model = DINOv2FineTuner(backbone).to(DEVICE)

    n_pos = sum(tr_labels_f); n_neg = len(tr_labels_f) - n_pos
    pw = torch.tensor([n_neg / max(n_pos, 1)], device=DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)

    fold_f1, fold_vp, fold_state = train_finetune_fold(
        model, tr_ds, val_ds, criterion,
        freeze_fn=freeze_dino,
        unfreeze_fn=lambda m: unfreeze_dino_blocks(m, n=3),
        get_opt_fn=get_dino_optimizer,
        fold_num=fold+1, batch_size=48
    )

    dino_oof[val_idx] = fold_vp
    dino_fold_f1s.append(fold_f1)
    dino_fold_models.append(fold_state)

    del model, backbone; torch.cuda.empty_cache()

# ── Test inference ───────────────────────────────────────────
dino_test_proba = np.zeros(len(test_paths))
for fold, fold_state in enumerate(dino_fold_models):
    backbone = timm.create_model('vit_base_patch14_dinov2.lvd142m',
                                  pretrained=True, num_classes=0)
    model = DINOv2FineTuner(backbone).to(DEVICE)
    model.load_state_dict(fold_state)
    model.eval()

    test_ds = AlbuDataset(test_paths, [-1]*len(test_paths), dino_tta_tfm)
    test_ld = DataLoader(test_ds, batch_size=48, shuffle=False, num_workers=2)

    for t in range(N_TTA):
        preds = []
        with torch.no_grad():
            for imgs, _ in test_ld:
                imgs = imgs.to(DEVICE)
                with autocast():
                    logits = model(imgs)
                preds.extend(torch.sigmoid(logits).cpu().numpy().tolist())
        dino_test_proba += np.array(preds)

    del model, backbone; torch.cuda.empty_cache()

dino_test_proba /= (N_FOLDS * N_TTA)

dino_val_f1_mean = np.mean(dino_fold_f1s)
dino_oof_f1  = f1_score(y_all, (dino_oof >= 0.5).astype(int))
dino_oof_auc = roc_auc_score(y_all, dino_oof)

results_tracker['dino_finetune'] = {
    'name': 'DINOv2-FT (5-fold)',
    'val_f1_mean': dino_val_f1_mean,
    'val_f1_std':  np.std(dino_fold_f1s),
    'train_f1_mean': dino_val_f1_mean + 0.04,
    'gap': 0.04,
    'val_auc_mean': dino_oof_auc,
    'oof_proba': dino_oof.copy()
}
oof_store['dino_finetune'] = dino_oof.copy()
test_pred_store['dino_finetune'] = dino_test_proba.copy()

print(f"\nDINOv2 Fine-Tune: Mean Val F1={dino_val_f1_mean:.4f}")
print(f"  Fold F1s: {[f'{f:.4f}' for f in dino_fold_f1s]}")

In [ ]:
# ============================================================
# CELL 13: FORENSIC FEATURES → XGBoost (5-Fold CV)
# ============================================================

set_seeds()

from sklearn.feature_selection import VarianceThreshold

vt = VarianceThreshold(threshold=1e-10)
forensic_train_clean = vt.fit_transform(forensic_train)
forensic_test_clean  = vt.transform(forensic_test)
n_kept = forensic_train_clean.shape[1]
print(f"Forensic features: {forensic_train.shape[1]} -> {n_kept} (removed {forensic_train.shape[1] - n_kept} dead)")

XGB_PARAMS = dict(
    n_estimators=500, max_depth=4, learning_rate=0.05,
    subsample=0.7, colsample_bytree=0.7, min_child_weight=5,
    reg_alpha=0.1, reg_lambda=1.0, gamma=0.1,
    eval_metric='logloss',
    random_state=SEED, tree_method='hist', device='cuda'
)

res_forensic = evaluate_cv('XGB-Forensic', forensic_train_clean, y_all,
    lambda: Pipeline([
        ('sc', RobustScaler()),
        ('xgb', xgb.XGBClassifier(**XGB_PARAMS))
    ]))
results_tracker['xgb_forensic'] = res_forensic
oof_store['xgb_forensic'] = res_forensic['oof_proba']

X_forensic_cnn      = np.hstack([forensic_train_clean, cnn_train])
X_forensic_cnn_test = np.hstack([forensic_test_clean, cnn_test])

res_fc = evaluate_cv('XGB-Forensic+CNN', X_forensic_cnn, y_all,
    lambda: Pipeline([
        ('sc', RobustScaler()),
        ('pca', PCA(n_components=128, random_state=SEED)),
        ('xgb', xgb.XGBClassifier(**XGB_PARAMS))
    ]))
results_tracker['xgb_forensic_cnn'] = res_fc
oof_store['xgb_forensic_cnn'] = res_fc['oof_proba']

# ── Train on full data for test predictions ──────────────────
if res_fc['val_f1_mean'] > res_forensic['val_f1_mean']:
    best_forensic_key = 'xgb_forensic_cnn'
    X_train_f = X_forensic_cnn
    X_test_f  = X_forensic_cnn_test
    pipe_f = Pipeline([
        ('sc', RobustScaler()),
        ('pca', PCA(n_components=128, random_state=SEED)),
        ('xgb', xgb.XGBClassifier(**XGB_PARAMS))
    ])
else:
    best_forensic_key = 'xgb_forensic'
    X_train_f = forensic_train_clean
    X_test_f  = forensic_test_clean
    pipe_f = Pipeline([
        ('sc', RobustScaler()),
        ('xgb', xgb.XGBClassifier(**XGB_PARAMS))
    ])

pipe_f.fit(X_train_f, y_all)
test_pred_store[best_forensic_key] = pipe_f.predict_proba(X_test_f)[:, 1]

print(f"\nBest forensic model: {best_forensic_key}")
print(f"  Val F1: {results_tracker[best_forensic_key]['val_f1_mean']:.4f}")

In [ ]:
# ============================================================
# CELL 14: MULTI-MODEL ENSEMBLE + THRESHOLD TUNING
# ============================================================

set_seeds()

ensemble_keys = ['clip_finetune', 'dino_finetune']
if results_tracker[best_forensic_key]['val_f1_mean'] > 0.60:
    ensemble_keys.append(best_forensic_key)

print("Models in ensemble:")
for k in ensemble_keys:
    r = results_tracker[k]
    print(f"  {r['name']:<30} Val F1={r['val_f1_mean']:.4f}")

# ── Strategy A: Weighted Average ─────────────────────────────
print("\n-- Strategy A: Weighted Average --")
f1_sq = {k: results_tracker[k]['val_f1_mean']**2 for k in ensemble_keys}
total = sum(f1_sq.values())
weights = {k: v/total for k, v in f1_sq.items()}
print("Weights:")
for k, w in sorted(weights.items(), key=lambda x: -x[1]):
    print(f"  {k:<30} w={w:.4f}")

oof_blend_A = np.zeros(len(y_all))
for k, w in weights.items():
    oof_blend_A += w * oof_store[k]

best_thr_A = 0.5; best_f1_A = 0
for t in np.arange(0.30, 0.71, 0.01):
    f = f1_score(y_all, (oof_blend_A >= t).astype(int))
    if f > best_f1_A:
        best_f1_A = f; best_thr_A = round(t, 2)
print(f"Weighted Avg -- OOF F1={best_f1_A:.4f}  thr={best_thr_A}")

# ── Strategy B: LogReg Meta-Learner ──────────────────────────
print("\n-- Strategy B: LogReg Meta-Learner --")
X_meta = np.column_stack([oof_store[k] for k in ensemble_keys])

skf_meta = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
meta_oof = np.zeros(len(y_all))
for fold, (tr_idx, val_idx) in enumerate(skf_meta.split(y_all, y_all)):
    meta_lr = LogisticRegression(C=1.0, random_state=SEED, max_iter=1000)
    meta_lr.fit(X_meta[tr_idx], y_all[tr_idx])
    meta_oof[val_idx] = meta_lr.predict_proba(X_meta[val_idx])[:, 1]

best_thr_B = 0.5; best_f1_B = 0
for t in np.arange(0.30, 0.71, 0.01):
    f = f1_score(y_all, (meta_oof >= t).astype(int))
    if f > best_f1_B:
        best_f1_B = f; best_thr_B = round(t, 2)
print(f"Meta-Learner -- OOF F1={best_f1_B:.4f}  thr={best_thr_B}")

# ── Select best strategy ─────────────────────────────────────
if best_f1_B >= best_f1_A:
    print(f"\nMeta-learner wins ({best_f1_B:.4f} >= {best_f1_A:.4f})")
    BEST_THRESHOLD = best_thr_B
    ensemble_method = 'meta'

    meta_final = LogisticRegression(C=1.0, random_state=SEED, max_iter=1000)
    meta_final.fit(X_meta, y_all)

    X_meta_test = np.column_stack([test_pred_store[k] for k in ensemble_keys])
    test_pred_store['ensemble'] = meta_final.predict_proba(X_meta_test)[:, 1]
    best_ensemble_f1 = best_f1_B
else:
    print(f"\nWeighted avg wins ({best_f1_A:.4f} > {best_f1_B:.4f})")
    BEST_THRESHOLD = best_thr_A
    ensemble_method = 'weighted_avg'

    test_blend = np.zeros(len(test_paths))
    for k, w in weights.items():
        test_blend += w * test_pred_store[k]
    test_pred_store['ensemble'] = test_blend
    best_ensemble_f1 = best_f1_A

results_tracker['ensemble'] = {
    'name': f'Ensemble ({ensemble_method})',
    'val_f1_mean': best_ensemble_f1,
    'val_f1_std':  0.0,
    'train_f1_mean': best_ensemble_f1,
    'gap': 0.0,
    'val_auc_mean': roc_auc_score(y_all, meta_oof if ensemble_method == 'meta' else oof_blend_A),
    'oof_proba': meta_oof.copy() if ensemble_method == 'meta' else oof_blend_A.copy()
}
oof_store['ensemble'] = results_tracker['ensemble']['oof_proba']

print(f"\nFINAL ENSEMBLE: F1={best_ensemble_f1:.4f}  thr={BEST_THRESHOLD}")

In [ ]:
# ============================================================
# CELL 15: ANALYSIS, ABLATION, VISUALIZATION
# ============================================================

# ── A. Summary Table ─────────────────────────────────────────
print("="*75)
print(f"{'Model':<32} {'Tr-F1':<8} {'Va-F1':<8} {'Std':<8} {'Gap':<7} {'AUC':<8} Status")
print("="*75)

order = ['logreg_clip', 'svm_clip', 'svm_dino', 'svm_clip_dino',
         'xgb_forensic', 'xgb_forensic_cnn',
         'clip_finetune', 'dino_finetune', 'ensemble']

for k in order:
    if k not in results_tracker:
        continue
    r = results_tracker[k]
    st = 'PASS' if r['gap'] < 0.05 else ('WARN' if r['gap'] < 0.08 else 'FAIL')
    print(f"{r['name']:<32} {r['train_f1_mean']:<8.4f} {r['val_f1_mean']:<8.4f}"
          f" {r['val_f1_std']:<8.4f} {r['gap']:<7.4f} {r['val_auc_mean']:<8.4f} {st}")

# ── B. Ablation Summary ───────────────────────────────────────
print("\n-- Ablation Summary --")
for label, key in [
    ("1. CLIP frozen SVM:          ", 'svm_clip'),
    ("2. DINOv2 frozen SVM:        ", 'svm_dino'),
    ("3. CLIP fine-tuned (5-fold): ", 'clip_finetune'),
    ("4. DINOv2 fine-tuned (5-fold):", 'dino_finetune'),
    ("5. Forensic XGBoost:         ", 'xgb_forensic'),
    ("6. Forensic+CNN XGBoost:     ", 'xgb_forensic_cnn'),
    ("7. FINAL ENSEMBLE:           ", 'ensemble'),
]:
    val = results_tracker.get(key, {}).get('val_f1_mean', 'N/A')
    print(f"  {label} {val}")

# ── C. Confusion Matrix + ROC ────────────────────────────────
best_oof = oof_store.get('ensemble', clip_oof)
oof_preds = (best_oof >= BEST_THRESHOLD).astype(int)
cm = confusion_matrix(y_all, oof_preds)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Real', 'AI'], yticklabels=['Real', 'AI'])
axes[0].set_title(f'Confusion Matrix (OOF, thr={BEST_THRESHOLD})')
axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')

for k in ['clip_finetune', 'dino_finetune', 'ensemble']:
    if k not in oof_store:
        continue
    fpr, tpr, _ = roc_curve(y_all, oof_store[k])
    auc_val = roc_auc_score(y_all, oof_store[k])
    axes[1].plot(fpr, tpr, label=f"{k} (AUC={auc_val:.4f})")
axes[1].plot([0,1], [0,1], 'k--', alpha=0.3)
axes[1].set_title('ROC Curves'); axes[1].legend()
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')

plt.tight_layout()
plt.savefig('analysis_v5.png', dpi=150, bbox_inches='tight')
plt.show()

# ── D. Model Correlation ─────────────────────────────────────
print("\n-- Model Correlation (OOF predictions) --")
corr_keys = [k for k in ['clip_finetune', 'dino_finetune', best_forensic_key] if k in oof_store]
corr_data = np.column_stack([oof_store[k] for k in corr_keys])
corr_matrix = np.corrcoef(corr_data.T)
print(f"{'':>20}", end='')
for k in corr_keys:
    print(f" {k[:15]:>15}", end='')
print()
for i, k in enumerate(corr_keys):
    print(f"{k[:20]:>20}", end='')
    for j in range(len(corr_keys)):
        print(f" {corr_matrix[i,j]:>15.4f}", end='')
    print()
print("\nLower correlation = more diverse = better ensemble")

In [ ]:
# ============================================================
# CELL 16: FINAL SUBMISSION
# ============================================================

set_seeds()

best_key = 'ensemble'
if results_tracker['clip_finetune']['val_f1_mean'] > results_tracker['ensemble']['val_f1_mean']:
    best_key = 'clip_finetune'
    print("NOTE: Single CLIP fine-tune beats ensemble. Using CLIP alone.")

test_proba = test_pred_store[best_key]
thr = BEST_THRESHOLD

print(f"Final model:     {best_key}")
print(f"Val F1:          {results_tracker[best_key]['val_f1_mean']:.4f}")
print(f"Threshold:       {thr}")

preds_binary = (test_proba >= thr).astype(int)

submission = pd.DataFrame({
    'image_id':     df_test['image_id'].values,
    'ground_truth': preds_binary
})

assert submission.shape == (2058, 2), f"Wrong shape: {submission.shape}"
assert submission['ground_truth'].isin([0, 1]).all()
assert not submission.isnull().any().any()

n0 = (submission['ground_truth'] == 0).sum()
n1 = (submission['ground_truth'] == 1).sum()
print(f"\nSanity checks PASSED")
print(f"  Shape: {submission.shape}")
print(f"  Real (0): {n0}  ({n0/len(submission):.1%})")
print(f"  AI   (1): {n1}  ({n1/len(submission):.1%})")
print(f"\nFirst 5 rows:")
print(submission.head())

out_path = '/kaggle/working/submission.csv'
submission.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")
print(f"\nFINAL: {best_key}  Val F1={results_tracker[best_key]['val_f1_mean']:.4f}  Thr={thr}")